In [ ]:
# Setup (added for the release): the notebook's folder on sys.path, then work in its data folder.
# The curation ran with the current directory = that folder; see curation/README.md for its contents.
import os, sys
if "data/curation/" not in os.getcwd().replace(os.sep, "/") + "/":
    sys.path.insert(0, os.path.abspath("."))
    os.chdir('../../data/curation/rBMRB')
print("working folder:", os.getcwd())


# PANAV re-referencing

Needs makeshift with `makeshift_panav.patch` (`curation/README.md`, Tools).

In [ ]:
import sys
MAKESHIFT_ROOT = "../../../external/makeshift"
if MAKESHIFT_ROOT not in sys.path:
    sys.path.insert(0, MAKESHIFT_ROOT)

import makeshift as ms
from makeshift.reref.panav import reref_panav
import os
import glob
import json
import tqdm
import pickle
import numpy as np
import pandas as pd  # NEW

base_path = os.getcwd()
data_path = os.path.join(base_path, "bmrb_entries")
atom_types = ['H','HA','N','CA','CB', 'C']
result_dict = {atom: {} for atom in atom_types}
result_dict.update({atom + "_mask": {} for atom in atom_types})
result_dict.update({atom + "_flag": {} for atom in atom_types})
result_dict["entry_ID"] = {}
result_dict["sequence"] = {}
result_dict["offset"] = {}

idx = 1
out_of_bounds = set()
mismatch = set()
notfit_refer = set()

three_to_one = {
    'ALA': 'A', 'ARG': 'R', 'ASN': 'N', 'ASP': 'D', 'CYS': 'C',
    'GLN': 'Q', 'GLU': 'E', 'GLY': 'G', 'HIS': 'H', 'ILE': 'I',
    'LEU': 'L', 'LYS': 'K', 'MET': 'M', 'PHE': 'F', 'PRO': 'P',
    'SER': 'S', 'THR': 'T', 'TRP': 'W', 'TYR': 'Y', 'VAL': 'V'
    }
one_to_three = {v: k for k, v in three_to_one.items()}  # NEW

def get_cs_seq(cs):
    df_unique = cs.drop_duplicates(subset='Seq_ID')
    cs_id = df_unique['Seq_ID'].astype(int).tolist()
    seq_single_letter = df_unique['Comp_ID'].map(three_to_one)
    sequence = ''.join(seq_single_letter.dropna())
    return sequence, cs_id

# NEW
def fill_missing_residues(df, sequence):
    """Add NaN-valued placeholder rows for residues absent from df, so reref_panav
    can look up the i-1 residue identity. Returns df unchanged if Seq_ID numbering
    does not match the sequence."""
    seen = df.drop_duplicates('Seq_ID')
    for sid, comp in zip(seen['Seq_ID'], seen['Comp_ID']):
        if not (1 <= sid <= len(sequence)) or three_to_one.get(comp) != sequence[sid - 1].upper():
            return df
    present = set(seen['Seq_ID'])
    fill = [
        {'Seq_ID': i, 'Comp_ID': one_to_three[aa.upper()], 'Atom_ID': 'N', 'Val': np.nan}
        for i, aa in enumerate(sequence, start=1)
        if i not in present and aa.upper() in one_to_three
    ]
    return pd.concat([df, pd.DataFrame(fill)], ignore_index=True) if fill else df

with open(os.path.join(base_path, "bmrb_cs_data_raw_stage2_noph_notemp.pkl"), 'rb') as f:
    rawdata = pickle.load(f)

n_filled = 0  # NEW: number of entries that received placeholder rows
processed = 0
for idx, name in enumerate(tqdm.tqdm(list(rawdata.keys())), start=1):
        folder_name = name.split("_")[0]
        folder_path = os.path.join(data_path, folder_name)

        str_files = glob.glob(os.path.join(folder_path, "*_3.str"))

        df = rawdata[name]["chemical shifts"]

        key = f"{idx}"

        df['Seq_ID'] = df['Seq_ID'].astype(int)
        df = df.sort_values(by='Seq_ID').reset_index(drop=True)

        cs_sequence, cs_dict = get_cs_seq(df)

        # MOVED: read the sequence before reref_panav so missing residues can be filled
        entities = ms.NMRStarEntry.from_file(str_files[0]).sequences()
        sequence = entities['Polymer_seq_one_letter_code'].values[0]

        # NEW: placeholder rows only go into the df passed to reref_panav; df itself is unchanged
        df_reref = fill_missing_residues(df, sequence)
        if len(df_reref) > len(df):
            n_filled += 1
        # Needs curation/rBMRB/makeshift_panav.patch (curation/README.md, Tools): with cona=False the unpatched
        # function returns None instead of the deviant and suspicious lists used below.
        offsets, check, cona = reref_panav(df_reref, cona=False)

        df = df.loc[
            df.Atom_ID.isin(['H', 'HA', 'N', 'CA', 'CB', 'C']) | df.Atom_ID.str.contains('^HA', na=False)
        ].copy()
        df['Atom_ID'] = df['Atom_ID'].replace({'HA2': 'HA', 'HA3': 'HA'})
        df['Val'] = df.groupby(['Seq_ID', 'Atom_ID'])['Val'].transform('mean')
        df = df.drop_duplicates(subset=['Seq_ID', 'Atom_ID'])

        if not df.empty:

            all_zero = all(value == False for value in check.values())
            if all_zero:
                notfit_refer.add(name)
                continue

            result_dict["entry_ID"][key] = name
            # reref_panav does not return a corrected DataFrame directly (unlike ReRef.rereference),
            # so we apply the returned offsets ourselves.
            invalid_set = {(d['seq_id'], d['atom']) for d in cona.get('deviant', [])} | \
                           {(d['seq_id'], d['atom']) for d in cona.get('suspicious', [])}

            df['Val'] = df.apply(
                lambda row: row['Val'] + offsets[row['Atom_ID']]
                if offsets.get(row['Atom_ID']) is not None else row['Val'],
                axis=1,
            )

            # (the entities / sequence lines were moved above)
            result_dict["sequence"][key] = sequence

            for atom in atom_types:
                cs_atom = np.full(len(sequence), None, dtype=object)
                cs_atom_flag = np.full(len(sequence), False, dtype=bool)
                sub_df = df[df['Atom_ID'] == atom]
                for _, row in sub_df.iterrows():
                    seq_idx_cs = int(row['Seq_ID'])
                    try:
                        if three_to_one[row["Comp_ID"]] != sequence[seq_idx_cs - 1].upper():
                            print(f"Residue mismatch at Seq_ID {row['Seq_ID']} in {name}")
                            mismatch.add(name)
                        elif (seq_idx_cs, atom) in invalid_set:
                            cs_atom[seq_idx_cs - 1] = None
                            cs_atom_flag[seq_idx_cs - 1] = True
                        else:
                            cs_atom[seq_idx_cs - 1] = row['Val']
                    except IndexError:
                        print(f"Seq_ID {row['Seq_ID']} out of bounds in {name}")
                        out_of_bounds.add(name)

                # back at the indentation of sub_df = ...: runs once per atom type,
                # whether sub_df has 0, 1 or more rows.
                if atom in check and not check[atom]:
                    cs_atom[:] = None

                cs_atom_mask = np.array([v is not None for v in cs_atom], dtype=bool)

                result_dict[atom][key] = cs_atom.tolist()
                result_dict[atom + "_mask"][key] = cs_atom_mask.tolist()
                result_dict[atom + "_flag"][key] = cs_atom_flag.tolist()

            result_dict["offset"][key] = offsets  # simplified: the old loop wrote the same value 4 times

            processed += 1

with open(os.path.join(base_path, "bmrb_cs_data_withoffset_panav.json"), 'w') as f:
    json.dump(result_dict, f, indent=4)

print(f"Processed {processed} entries, results saved to bmrb_cs_data_withoffset_panav.json")
print(f"Filled missing residues for i-1 lookup in {n_filled} entries")  # NEW
print(f"There are {len(out_of_bounds)} entries with out-of-bounds Seq_IDs:", out_of_bounds)
print(f"There are {len(mismatch)} entries with residue mismatches:", mismatch)
print(f"There are {len(notfit_refer)} entries with no N/CA/CB/C offset fitted:", notfit_refer)

In [ ]:
import json
json_path = "bmrb_cs_data_withoffset_panav.json"
with open(json_path, 'r') as f:
    data = json.load(f)

dictionary = {v: k for k, v in data['entry_ID'].items()}

no_entries = ['bmr52649_1', 'bmr51689_1', 'bmr50824_1', 'bmr50826_1', 'bmr50825_1', 'bmr18265_1', 'bmr19825_1', 'bmr27587_1', 'bmr26648_1']
for entry in no_entries:
    key = dictionary.get(entry)
    if key in data['entry_ID'].keys():
        for big_key in data.keys():
            del data[big_key][key]  # Remove the entry with flag for now, can be further processed later

json_path_new = "bmrb_cs_data_withoffset_panav_v1.json"
with open(json_path_new, 'w') as f:
    json.dump(data, f, indent=4)
print(len(data['entry_ID']))

In [ ]:
import json

src = "bmrb_cs_data_withoffset_panav_v1.json"
dst = "bmrb_cs_data_withoffset_panav_v1_flags_only.json"

with open(src) as f:
    data = json.load(f)

filtered = {
    key: value
    for key, value in data.items()
    if key == "entry_ID" or key.endswith("_flag")
}

with open(dst, "w") as f:
    json.dump(filtered, f, indent=4)

print("kept keys:", list(filtered.keys()))

In [ ]:
import json
from pathlib import Path

SRC = Path("bmrb_cs_data_withoffset_panav_v1_flags_only.json")
DST = Path("bmrb_cs_data_withoffset_panav_v2_flags_only.json")

with open(SRC) as f:
    data = json.load(f)

data["entry_ID"] = {
    ref_id: entry_id.removeprefix("bmr")
    for ref_id, entry_id in data["entry_ID"].items()
}

with open(DST, "w") as f:
    json.dump(data, f)

print(f"Wrote {len(data['entry_ID'])} entry_ID values (prefix stripped) to {DST}")